In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import lingam
import networkx as nx
import matplotlib.pyplot as plt

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
cp_root = project_root / "data" / "raw" / "CausalPitfallsData"
#test it works
cp_root

#path to put results
output_dir = project_root/"results"/"graphs_LiM"
output_dir.mkdir(parents=True,exist_ok=True)

In [3]:
"""LiM expects a numeric matrix with continuous and discrete columns as well as
an array per variable indicating 0 for discrete and 1 for continuous.
Similar to what we did for DAGBagM, we write a helper function to infer the datatype."""

def infer_type(df):
    """
    Infers a 'flag array' for a given dataframe.
    0 corresponds to discrete variables, 1 to continuous variables
    """

    flags=[]
    for col in df.columns:
        x = df[col].dropna()
        #classify numeric columns
        if np.issubdtype(x.dtype, np.number):
            vals = x.unique()
            if len(vals) <= 1:
                flags.append(0) #variable is discrete
            else:
                flags.append(1) # variable is continuous
    return np.asarray(flags, dtype=int)

def run_lim(df, seed=1):
    """
    Clean dataframe and generate the numeric matrix as well as flag array.
    Run LiM on the data.
    """

    df_clean=df.dropna().copy()
    flags = infer_type(df_clean)
    flags_2d = flags.reshape(1,-1)
    cols = list(df_clean.columns)

    X = df_clean.to_numpy(dtype=float)

    model = lingam.LiM()

    model.fit(X, flags_2d, only_global=True)
    #LiM creates an adjacency matrix
    M = model.adjacency_matrix_
    #coerce adjacency matrix into a matrix of 0's and 1's
    adj = (np.abs(M) > 0).astype(int)

    return adj, cols

In [4]:
def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [5]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [6]:
def one_hot_encode(df):
    non_numeric_cols = df.select_dtypes(exclude=["number"]).columns

    #return the same df if no non numeric
    if len(non_numeric_cols) == 0:
        return df.copy()

    df_encoded = pd.get_dummies(
        df,
        columns=list(non_numeric_cols),
        drop_first=False,   # keep full one-hot encoding [web:195][web:198]
        dtype=int,
    )
    return df_encoded

In [7]:
csv_files = sorted(cp_root.rglob("*.csv"))

for csv_path in csv_files:
    #relative path
    rel = csv_path.relative_to(cp_root)
    print(f"Processing: {rel}")

    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            print("  Skipped (empty file)")
            continue
        df_enc = one_hot_encode(df)
        # Run LiM on this dataset
        adj,nodes = run_lim(df_enc, seed=1)

        #Output schema scenario__file__LiM.png
        parts = rel.parts           
        scenario = parts[0] if len(parts) > 1 else "root"
        name_no_ext = csv_path.stem

        out_name = f"{scenario}__{name_no_ext}__LiM.png"
        out_path = output_dir / out_name

        # Save PNG
        #draw_graph(adj, nodes, out_path)
        draw_graphviz_dag(adj, out_path, nodes)
        print(f"  Saved graph to {out_path.relative_to(project_root)}")

    except Exception as e:
        print(f"  ERROR on {rel}: {e}")

Processing: berkson_paradox/.ipynb_checkpoints/admission_bias-checkpoint.csv
W_est (without the 2nd phase) is: 
 [[0.        0.       ]
 [0.3279654 0.       ]]
  Saved graph to results/graphs_LiM/berkson_paradox__admission_bias-checkpoint__LiM.png
Processing: berkson_paradox/.ipynb_checkpoints/loan_approval_bias-checkpoint.csv
W_est (without the 2nd phase) is: 
 [[0.         0.51445447]
 [0.         0.        ]]
  Saved graph to results/graphs_LiM/berkson_paradox__loan_approval_bias-checkpoint__LiM.png
Processing: berkson_paradox/.ipynb_checkpoints/movie_success_bias-checkpoint.csv
W_est (without the 2nd phase) is: 
 [[0.         0.16718244]
 [0.         0.        ]]
  Saved graph to results/graphs_LiM/berkson_paradox__movie_success_bias-checkpoint__LiM.png
Processing: berkson_paradox/admission_bias.csv
W_est (without the 2nd phase) is: 
 [[0.        0.       ]
 [0.3279654 0.       ]]
  Saved graph to results/graphs_LiM/berkson_paradox__admission_bias__LiM.png
Processing: berkson_parad

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)
/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/numpy/linalg/linalg.py:677: RuntimeWarning: overflow encountered in matmul
  z = a if z is None else fmatmul(z, z)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:251: RuntimeWarning: invalid value encountered in multiply
  h = (G_h.T * M).sum() - d
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:117: RuntimeWarning: invalid value encountered in logaddexp
  (np.logaddexp(0, M) - X * M) * np.absolute(dis_con - 1)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:263: RuntimeWarning: overflow encountered in scalar multiply
  obj = loss + 0.5 * rho * h * h + alpha * h + self._lambda1 * w.sum()
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:265: RuntimeWarning: overflow encountered in

W_est (without the 2nd phase) is: 
 [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 5.01585718e+02  0.00000000e+00  0.00000000e+00  1.65263068e-01
   0.00000000e+00  0.00000000e+00 -5.12338612e-01]
 [ 2.47206194e+02  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 4.70058193e+02 -1.84669584e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  9.06311208e+00]
 [ 2.52722142e+02  0.00000000e+00  5.36143881e-01  1.55118424e-01
   0.00000000e+00 -2.56108731e+00 -2.04788881e+00]
 [ 4.66508787e+02  2.83466872e+00  0.00000000e+00  4.48151456e-01
   0.00000000e+00  0.00000000e+00 -2.21812037e+00]
 [ 0.00000000e+00  0.00000000e+00 -3.55254739e+01  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]]
  Saved graph to results/graphs_LiM/casual_effect__device_failure_data__LiM.png
Processing: casual_effect/machine_maintenance_data.csv
W_est (wi

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:251: RuntimeWarning: overflow encountered in multiply
  h = (G_h.T * M).sum() - d
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:263: RuntimeWarning: invalid value encountered in scalar multiply
  obj = loss + 0.5 * rho * h * h + alpha * h + self._lambda1 * w.sum()
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:265: RuntimeWarning: invalid value encountered in multiply
  G_smooth = G_loss + (rho * h + alpha) * G_h.T * W * 2  # 2019
/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:117: RuntimeWarning: invalid value encountered in logaddexp
  (np.logaddexp(0, M) - X * M) * np.absolute(dis_con - 1)


W_est (without the 2nd phase) is: 
 [[   0.            0.          -42.3688705     0.        ]
 [-147.48877281    0.         -285.77544588    0.        ]
 [   0.            0.            0.            0.        ]
 [   0.         7036.74959786 -130.6733728     0.        ]]
  Saved graph to results/graphs_LiM/causal_direction_iv__ecommerce_sem__LiM.png
Processing: causal_direction_iv/environment_sem.csv
W_est (without the 2nd phase) is: 
 [[0.         0.85858714 0.         0.30911888]
 [0.         0.         0.         1.1995806 ]
 [1.99870051 0.16087455 0.         0.18303971]
 [0.         0.         0.         0.        ]]
  Saved graph to results/graphs_LiM/causal_direction_iv__environment_sem__LiM.png
Processing: causal_direction_iv/marketing_sem.csv
W_est (without the 2nd phase) is: 
 [[ 0.          0.          0.          0.        ]
 [-0.15780538  0.         -2.15468659  2.07482893]
 [ 1.03819663  0.          0.          0.        ]
 [ 0.12241008  0.          1.66290189  0.        

...

W_est (without the 2nd phase) is: 
 [[  0.           0.           0.           0.        ]
 [  0.           0.           1.88534777 -15.27087155]
 [  4.93785593  -0.98310755   0.           5.23291683]
 [152.03594638   0.54368295   3.16919727   0.        ]]
  Saved graph to results/graphs_LiM/temporal_stability__temporal_variant3__LiM.png
Processing: temporal_stability/temporal_variant4.csv
W_est (without the 2nd phase) is: 
 [[ 0.00000000e+00 -1.21349843e-01  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00 -7.85435464e+01  0.00000000e+00  0.00000000e+00]
 [ 1.47962136e+02  4.68843673e+01  1.21883307e+00  0.00000000e+00]]
  Saved graph to results/graphs_LiM/temporal_stability__temporal_variant4__LiM.png
Processing: treatment_mediator/economic_stimulus.csv


/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[  0.           2.96485633 127.16854677   0.           1.8541593
    0.        ]
 [  0.           0.           2.39891724   0.           0.
    2.01572032]
 [  0.           0.           0.           0.           0.
    0.        ]
 [  0.         -10.90506357  -0.27591107   0.           0.39949457
    2.52038308]
 [  0.          23.52020425   0.           0.           0.
    2.49041024]
 [  0.           0.           0.           0.           0.
    0.        ]]
  Saved graph to results/graphs_LiM/treatment_mediator__economic_stimulus__LiM.png
Processing: treatment_mediator/education_subsidy_policy.csv
W_est (without the 2nd phase) is: 
 [[  0.           0.           0.         -15.25129316   0.
    1.50576635]
 [  0.15678123   0.           0.          25.27104582   0.19056602
    4.79139109]
 [  0.           1.83287824   0.          -4.43442282  90.04404
    0.        ]
 [  0.           0.           0.           0.           2.09606688
    1.77328842

/dcs/23/u2200504/thesis/recidivism-causal/code/lingam/lingam/lim.py:119: RuntimeWarning: overflow encountered in cosh
  loss_con = np.sum(-np.log(np.cosh(R)) * dis_con)


W_est (without the 2nd phase) is: 
 [[  0.           0.           0.         -13.78359163   0.
    1.33736749]
 [  0.1249577    0.           0.          23.40870084   0.84026459
    3.65396274]
 [  0.           1.94298641   0.           3.12439408  21.62729115
    0.        ]
 [  0.           0.           0.           0.           7.82668851
    1.81858731]
 [  0.           0.           0.           0.           0.
    0.        ]
 [  0.           0.           0.           0.          -2.16489791
    0.        ]]
  Saved graph to results/graphs_LiM/treatment_mediator__infrastructure_spending_policy__LiM.png
Processing: treatment_mediator/job_training_policy.csv
W_est (without the 2nd phase) is: 
 [[ 0.          0.39415118  0.         -8.42877831  0.          0.        ]
 [ 0.          0.          0.         22.78106584  0.          6.88976115]
 [ 0.          1.84721363  0.          4.14110957 17.91829687  0.        ]
 [ 0.          0.          0.          0.          3.91869873  1.7588